### MI estimation with MINDE (MNIST)

In [13]:
import torch
import numpy as np

In [11]:
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [14]:
# set the random seed
torch.manual_seed(0)
np.random.seed(0)

In [15]:
# load the data
transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# load mnist
train_dataset = datasets.MNIST('../data/', train=True, download=True, transform=transform)

In [16]:
grouped_data = {label: [] for label in range(10)}
train_loaders = []

# group the data
for data in train_dataset:
    label = data[1]
    grouped_data[label].append(data)

class_batch_size = 100
for c in range(10):
    train_loaders.append(torch.utils.data.DataLoader(grouped_data[c], batch_size=class_batch_size, shuffle=True))

# construct the train dataset: X, Y are i.i.d. samples from the same class
X = []
Y = []
for c in range(10):
    # for _ in range(10):
    data1, _ = next(iter(train_loaders[c]))
    data2, _ = next(iter(train_loaders[c]))
    X.append(data1)
    Y.append(data2)

X = torch.cat(X)
Y = torch.cat(Y)

In [19]:
import jax
jax.config.update('jax_platforms', 'cpu')
# import bmi
from src.libs.minde import MINDE
from src.scripts.helper import get_data_loader, get_default_config

In [20]:
args = get_default_config()
args.type ="c"
args.test_epoch=10
args.max_epochs=200
args.warmup_epochs = 0
args.bs = 20
args.lr = 1e-3
args.arch = "mlp"
args.importance_sampling = True
args.use_ema = True

In [21]:
import torch
from torch.utils.data import Dataset, DataLoader

class InMemoryDictDataset(Dataset):
    def __init__(self, x, y):
        # x, y: numpy arrays or torch tensors
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
        if not isinstance(y, torch.Tensor):
            y = torch.tensor(y, dtype=torch.float32)
        # keep data on CPU for DataLoader; move to device inside training loop
        self.x = x.cpu().float()
        self.y = y.cpu().float()

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return {"x": self.x[idx], "y": self.y[idx]}

# usage
dataset = InMemoryDictDataset(X.reshape(1000,-1), Y.reshape(1000,-1))    # X, Y defined earlier
loader = DataLoader(dataset, batch_size=20, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)

In [24]:
model = MINDE(args,var_list={"x":1024,"y":1024}, gt = np.log(10))
model.fit(loader,loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4090 D') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type           | Params
---------------------------------------------
0 | score     | UnetMLP_simple | 1.7 M 
1 | model_ema | EMA            | 1.7 M 
---------------------------------------------
3.4 M     Trainable params
0         Non-trainable params
3.4 M     Total params
13.697    Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/star/anaconda3/envs/MINDE/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:492: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
/home/star/anaconda3/envs/MINDE/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.
/home/star/anaconda3/envs/MINDE/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Epoch:  10  GT:  2.303 MINDE_estimate:  29.699 MINDE_sigma_estimate:  95.67
Epoch:  20  GT:  2.303 MINDE_estimate:  72.268 MINDE_sigma_estimate:  95.703
Epoch:  30  GT:  2.303 MINDE_estimate:  57.129 MINDE_sigma_estimate:  5.519
Epoch:  40  GT:  2.303 MINDE_estimate:  35.887 MINDE_sigma_estimate:  -25.469


Validation: |          | 0/? [00:00<?, ?it/s]

... (3 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch:  150  GT:  2.303 MINDE_estimate:  16.644 MINDE_sigma_estimate:  -61.925
Epoch:  160  GT:  2.303 MINDE_estimate:  16.23 MINDE_sigma_estimate:  -61.631
Epoch:  170  GT:  2.303 MINDE_estimate:  17.458 MINDE_sigma_estimate:  -67.647
Epoch:  180  GT:  2.303 MINDE_estimate:  17.174 MINDE_sigma_estimate:  -67.948
Epoch:  190  GT:  2.303 MINDE_estimate:  17.156 MINDE_sigma_estimate:  -67.608


Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=200` reached.
